# Course 1, Week 1 — Getting Started with PyTorch

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #1](https://github.com/majorgilles/pytorch_for_deep_learning/issues/1)

**Focus:** Work with tensors, devices, operations, and PyTorch's core programming model.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

## 1. Prepare the training data

Each row is one delivery example. Both tensors have shape $[4, 1]$: four examples and one feature or target per example. Keeping the batch dimension explicit lets the linear layer process all examples together.


In [2]:
# Inputs: distance in miles, with shape [examples, features].
distances: torch.Tensor = torch.tensor(
    [[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32
)

# Targets: observed delivery time in minutes, with shape [examples, targets].
times: torch.Tensor = torch.tensor(
    [[6.96], [12.11], [16.77], [22.21]], dtype=torch.float32
)

## 2. Define the model and training objective

A single linear layer learns $\hat{y} = wx + b$, where $x$ is distance and $\hat{y}$ is the predicted delivery time. Mean squared error measures the average squared difference between predictions and targets:

$$\operatorname{MSE} = \frac{1}{N} \sum_{i=1}^{N}(\hat{y}_i-y_i)^2.$$


In [3]:
# Map one input feature to one output with a learnable weight and bias.
model: nn.Sequential = nn.Sequential(nn.Linear(in_features=1, out_features=1))

In [4]:
# Compare predictions with targets using mean squared error.
loss_function: nn.MSELoss = nn.MSELoss()

# Update the model parameters with stochastic gradient descent.
optimizer: optim.SGD = optim.SGD(model.parameters(), lr=0.01)

## 3. Train the model

Each iteration performs a forward pass, computes the loss, backpropagates gradients, and updates the parameters. Gradients must be cleared first because PyTorch accumulates them by default.


In [5]:
for epoch in range(500):
    # Clear gradients left over from the previous iteration.
    optimizer.zero_grad()

    # Forward pass: predict one delivery time for each distance.
    outputs: torch.Tensor = model(distances)
    loss: torch.Tensor = loss_function(outputs, times)

    # Backward pass: compute gradients, then adjust the weight and bias.
    loss.backward()
    optimizer.step()

## 4. Run inference

Inference only needs a forward pass. The `torch.no_grad()` context disables gradient tracking, reducing unnecessary memory and computation.


In [6]:
with torch.no_grad():
    # Shape [1, 1] means one example with one distance feature.
    test_distance: torch.Tensor = torch.tensor([[25.0]], dtype=torch.float32)
    predicted_time: torch.Tensor = model(test_distance)
    print(f"Predicted time for 25 miles: {predicted_time.item():.1f} minutes")

Predicted time for 25 miles: 127.7 minutes


## 5. Tensor fundamentals

Most tensor problems involve an unexpected **shape**, **data type**, or **device**. Inspecting these properties before passing data to a model usually makes the mismatch clear.

### Shapes and model inputs

For tabular data, the common layout is $[B, F]$, where $B$ is the batch size and $F$ is the number of features per example. A linear layer configured with `in_features=1` accepts any batch size, but every example must contain exactly one feature.


In [7]:
batch_size: int = distances.shape[0]
feature_count: int = distances.shape[1]
expected_features: int = model[0].in_features

print(f"shape={distances.shape}, batch={batch_size}, features={feature_count}")
print(f"model expects {expected_features} feature per example")

# This tensor has three features per example and does not fit the current model.
delivery_features: torch.Tensor = torch.tensor(
    [[1.0, 9.0, 0.0], [2.0, 14.0, 1.0]], dtype=torch.float32
)
print(f"multi-feature shape={delivery_features.shape}")

shape=torch.Size([4, 1]), batch=4, features=1
model expects 1 feature per example
multi-feature shape=torch.Size([2, 3])


### Data types

PyTorch infers integer and floating-point types from the input values. Neural-network parameters commonly use `torch.float32`, so specifying `dtype` or converting with `.float()` keeps inputs compatible. Mixed operations follow PyTorch's type-promotion rules.


In [8]:
integer_values: torch.Tensor = torch.tensor([1, 2, 3])
inferred_floats: torch.Tensor = torch.tensor([1.0, 2.0, 3.0])
explicit_floats: torch.Tensor = torch.tensor([1, 2, 3], dtype=torch.float32)
converted_floats: torch.Tensor = integer_values.float()
promoted_values: torch.Tensor = integer_values + inferred_floats

print(integer_values.dtype, inferred_floats.dtype)
print(explicit_floats.dtype, converted_floats.dtype, promoted_values.dtype)

torch.int64 torch.float32
torch.float32 torch.float32 torch.float32


### Creation and reshaping

Tensors can be created from Python data or with helpers such as `zeros`, `ones`, and `rand`. `unsqueeze()` adds a size-one dimension; `squeeze()` removes size-one dimensions. Check `.shape` before reshaping so that the batch and feature meanings stay explicit.

When interoperating with NumPy, `torch.from_numpy()` shares memory with the array, while `torch.tensor()` creates a copy.


In [9]:
zeros: torch.Tensor = torch.zeros((2, 3))
ones: torch.Tensor = torch.ones((2, 3))
random_values: torch.Tensor = torch.rand((2, 3))

single_distance: torch.Tensor = torch.tensor(25.0)  # Scalar: shape []
batched_distance: torch.Tensor = single_distance.unsqueeze(0).unsqueeze(0)
squeezed_distance: torch.Tensor = batched_distance.squeeze()

print(single_distance.shape, batched_distance.shape, squeezed_distance.shape)

torch.Size([]) torch.Size([1, 1]) torch.Size([])


### Indexing, slicing, and scalar values

Tensor indexing follows Python's indexing rules. An indexed result remains a tensor; use `.item()` only when it contains exactly one element and a Python scalar is required.


In [10]:
first_prediction: torch.Tensor = outputs[0]
first_three_predictions: torch.Tensor = outputs[:3]
first_prediction_value: float = first_prediction.item()
first_distance: float = delivery_features[0, 0].item()

print(first_prediction, first_three_predictions)
print(first_prediction_value, first_distance)

tensor([6.9679], grad_fn=<SelectBackward0>) tensor([[ 6.9679],
        [12.0002],
        [17.0325]], grad_fn=<SliceBackward0>)
6.9678635597229 1.0


## 6. Element-wise operations and broadcasting

PyTorch applies arithmetic operators element by element. For the single-neuron model, every distance uses the same learned weight and bias:

$$\hat{y}_i = w x_i + b.$$

The tensor expression has the same form as scalar arithmetic, but computes predictions for the whole batch at once.


In [11]:
# Squeeze the learned parameters into scalar tensors with shape [].
weight: torch.Tensor = model[0].weight.detach().squeeze()
bias: torch.Tensor = model[0].bias.detach().squeeze()

# Apply the same weight and bias to every distance in the batch.
manual_predictions: torch.Tensor = distances * weight + bias
with torch.no_grad():
    model_predictions: torch.Tensor = model(distances)

assert torch.allclose(manual_predictions, model_predictions)
print(manual_predictions)

tensor([[ 6.9678],
        [12.0001],
        [17.0325],
        [22.0648]])


### Broadcasting across a batch

Broadcasting avoids manually repeating values. PyTorch compares shapes from right to left; each pair of dimensions is compatible when the sizes are equal or one of them is $1$. Missing leading dimensions are also treated as size $1$.

A feature-adjustment tensor with shape $[3]$ therefore broadcasts across delivery data with shape $[B, 3]$. The same three adjustments are applied to every example without a Python loop.


In [12]:
# Scale distance by 1.1, leave hour unchanged, and penalize bad weather by 5.
feature_adjustments: torch.Tensor = torch.tensor([1.1, 1.0, 5.0])
adjusted_features: torch.Tensor = delivery_features * feature_adjustments

print(delivery_features.shape, feature_adjustments.shape)
print(adjusted_features)

torch.Size([2, 3]) torch.Size([3])
tensor([[ 1.1000,  9.0000,  0.0000],
        [ 2.2000, 14.0000,  5.0000]])


### Broadcasting across multiple dimensions

A row tensor with shape $[1, 3]$ and a column tensor with shape $[3, 1]$ broadcast to $[3, 3]$. The row is repeated downward and the column is repeated across. Shapes such as $[2, 3]$ and $[2, 2]$ are incompatible because their trailing dimensions are neither equal nor $1$.


In [13]:
row: torch.Tensor = torch.tensor([[1.0, 2.0, 3.0]])  # Shape [1, 3]
column: torch.Tensor = torch.tensor([[10.0], [20.0], [30.0]])  # Shape [3, 1]
broadcast_sum: torch.Tensor = row + column

print(f"{row.shape} + {column.shape} -> {broadcast_sum.shape}")
print(broadcast_sum)

torch.Size([1, 3]) + torch.Size([3, 1]) -> torch.Size([3, 3])
tensor([[11., 12., 13.],
        [21., 22., 23.],
        [31., 32., 33.]])
